<a href="https://colab.research.google.com/github/Mufliha-Hafsyah/Machine-Learning/blob/main/Week09/Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM PEMBELAJARAN MESIN**

**Nama**  : Mufliha Hafsyah Shahieza <br>
**NIM**   : 244107020147 <br>
**Kelas** : TI-3G

---

# **JS09 - Klasifikasi**<hr>
# **Tugas 2**

Buatlah model klasfikasi Multinomial Naive Bayes dengan ketentuan,
1. Menggunakan data spam.csv
2. Fitur CountVectorizer dengan mengaktifkan stop_words
3. Evaluasi hasilnya
<br>

Buatlah model klasfikasi Multinomial Naive Bayes dengan ketentuan,
1. Menggunakan data spam.csv
2. Fitur TF-IDF dengan mengaktifkan stop_words
3. Evaluasi hasilnya dan bandingkan dengan hasil pada Tugas no 2.
4. Berikan kesimpulan fitur mana yang terbaik pada kasus data spam.csv
<hr>

## **Langkah 1: Upload dataset**

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('spam.csv', encoding='latin-1')

print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())

(5572, 5)
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: o

**Cek kolom mana yang kabel, mana yang teks**

In [3]:
# sesuaikan nama kolom dengan hasil df.head() di atas
print(df.iloc[:, 0].value_counts())
print(df.iloc[:, 1].head(10).tolist())

v1
ham     4825
spam     747
Name: count, dtype: int64
['Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...', 'Ok lar... Joking wif u oni...', "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's", 'U dun say so early hor... U c already then say...', "Nah I don't think he goes to usf, he lives around here though", "FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, å£1.50 to rcv", 'Even my brother is not like to speak with me. They treat me like aids patent.', "As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune", 'WINNER!! As a valued network customer you have been selected to receivea å£900 prize reward! To claim call 09061701461. C

## **Langkah 2: Siapkan label dan split**

In [4]:
# Ambil hanya kolom label dan teks
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'text']
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Opsional: cek pesan duplikat (hanya dicatat, tidak dihapus)
print("Jumlah baris duplikat:", df.duplicated().sum())

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label_num'], test_size=0.2, random_state=42, stratify=df['label_num'])

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))

Jumlah baris duplikat: 403
(4457,) (1115,)
label_num
0    0.865829
1    0.134171
Name: proportion, dtype: float64


## **Langkah 3: Multinomial NB dengan CountVectorizer (nomor 1)**

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Vectorizer di-fit HANYA di data latih
cv_vec = CountVectorizer(stop_words='english')
X_train_cv = cv_vec.fit_transform(X_train)
X_test_cv = cv_vec.transform(X_test)
print("Ukuran matriks fitur (train):", X_train_cv.shape)

nb_cv = MultinomialNB()
nb_cv.fit(X_train_cv, y_train)
y_pred_cv = nb_cv.predict(X_test_cv)

print("Akurasi train:", accuracy_score(y_train, nb_cv.predict(X_train_cv)))
print("Akurasi test :", accuracy_score(y_test, y_pred_cv))
print(confusion_matrix(y_test, y_pred_cv))
print(classification_report(y_test, y_pred_cv, target_names=['ham', 'spam']))

Ukuran matriks fitur (train): (4457, 7440)
Akurasi train: 0.9946152120260264
Akurasi test : 0.9838565022421525
[[960   6]
 [ 12 137]]
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.96      0.92      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## **Langkah 4: Multinomial NB dengan TF-IDF (nomor 2)**

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorizer di-fit HANYA di data latih, split sama dengan model sebelumnya
tfidf_vec = TfidfVectorizer(stop_words='english')
X_train_tf = tfidf_vec.fit_transform(X_train)
X_test_tf = tfidf_vec.transform(X_test)
print("Ukuran matriks fitur (train):", X_train_tf.shape)

nb_tf = MultinomialNB()
nb_tf.fit(X_train_tf, y_train)
y_pred_tf = nb_tf.predict(X_test_tf)

print("Akurasi train:", accuracy_score(y_train, nb_tf.predict(X_train_tf)))
print("Akurasi test :", accuracy_score(y_test, y_pred_tf))
print(confusion_matrix(y_test, y_pred_tf))
print(classification_report(y_test, y_pred_tf, target_names=['ham', 'spam']))

Ukuran matriks fitur (train): (4457, 7440)
Akurasi train: 0.9845187345748261
Akurasi test : 0.968609865470852
[[966   0]
 [ 35 114]]
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



## **Langkah 5: Tabel perbandingan**

In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

hasil = pd.DataFrame({
    'Metrik': ['Akurasi test', 'Precision (spam)', 'Recall (spam)', 'F1 (spam)'],
    'CountVectorizer': [accuracy_score(y_test, y_pred_cv), precision_score(y_test, y_pred_cv),
                        recall_score(y_test, y_pred_cv), f1_score(y_test, y_pred_cv)],
    'TF-IDF': [accuracy_score(y_test, y_pred_tf), precision_score(y_test, y_pred_tf),
               recall_score(y_test, y_pred_tf), f1_score(y_test, y_pred_tf)]
})
print(hasil.round(4))

             Metrik  CountVectorizer  TF-IDF
0      Akurasi test           0.9839  0.9686
1  Precision (spam)           0.9580  1.0000
2     Recall (spam)           0.9195  0.7651
3         F1 (spam)           0.9384  0.8669


## **Hasil Evaluasi**

| Metrik | CountVectorizer | TF-IDF |
|---|---|---|
| Akurasi test | 0,9839 | 0,9686 |
| Precision (spam) | 0,9580 | 1,0000 |
| Recall (spam) | 0,9195 | 0,7651 |
| F1 (spam) | 0,9384 | 0,8669 |

## **Analisis Perbandingan**

Dataset spam.csv terdiri dari 5572 pesan dengan proporsi tidak seimbang (86,6% ham dan 13,4% spam), sehingga evaluasi tidak cukup memakai akurasi dan perlu melihat precision, recall, dan F1 pada kelas spam. Kedua model memakai pembagian data latih dan uji yang sama dengan stop words bahasa Inggris, sehingga perbedaan hasil berasal dari cara pembobotan fitur.

Model dengan CountVectorizer menghasilkan akurasi 0,9839 dengan recall spam 0,9195 (12 spam lolos dan 6 pesan normal salah diblokir). Model dengan TF-IDF menghasilkan precision spam sempurna (1,0000), yaitu tidak ada pesan normal yang salah diblokir, tetapi recall spam turun menjadi 0,7651 sehingga sekitar 35 spam lolos dari filter. Hal ini kemungkinan terjadi karena nilai TF-IDF berupa pecahan kecil, sehingga smoothing bawaan Multinomial Naive Bayes lebih dominan dan model cenderung memprediksi kelas mayoritas (ham). Perlu dicatat juga bahwa Multinomial Naive Bayes secara teori dirancang untuk data jumlah kata, sehingga CountVectorizer lebih sesuai dengan asumsinya.

Terdapat 403 baris duplikat pada dataset, dan 129 pesan uji juga muncul di data latih, sehingga akurasi kedua model kemungkinan sedikit lebih optimis. Namun karena keduanya memakai pembagian data yang sama, perbandingan tetap adil.

## **Kesimpulan**

Pada kasus data spam.csv, fitur **CountVectorizer** lebih baik karena menghasilkan akurasi, recall, dan F1 spam tertinggi (0,9839; 0,9195; 0,9384). TF-IDF unggul hanya pada precision (1,0000), sehingga lebih cocok jika prioritasnya adalah menghindari pesan normal yang salah diblokir, tetapi untuk tujuan menangkap spam secara menyeluruh CountVectorizer lebih efektif.